# Post-Cavity Calibrations Notebook

Calibration workflows using the **qubox_v2 v3** API with `CalibrationStore`
and calibration algorithms.

**Sections:**
0. OPX/Octave Mixer Calibration
1. Full Readout Calibration Pipeline
2. Pulse-Train Tomography & Correction
3. AllXY Gate Error Diagnostics
4. Fock-Resolved SQR Calibration

In [1]:
import sys
import numpy as np

sys.path.insert(0, r"E:\qubox")

from qualang_tools.units import unit

from qubox_v2.experiments.session import SessionManager
from qubox_v2.experiments import (
    CalibrateReadoutFull,
    ReadoutGEDiscrimination,
    ReadoutButterflyMeasurement,
    QubitPulseTrain,
    AllXY,
    FockResolvedPowerRabi,
    FockResolvedSpectroscopy,
)
from qubox_v2.calibration import (
    CalibrationStore,
    PulseTrainResult,
    FockSQRCalibration,
    fit_pulse_train,
    compute_corrected_knobs,
    fit_fock_sqr,
    optimize_fock_sqr_iterative,
    optimize_fock_sqr_spsa,
)

u = unit()

2026-02-21 13:46:19,071 - qm - INFO     - Starting session: a96bd537-5e5b-4dd2-8920-cff9628786a7


In [ ]:
experiment_path = r"E:\qubox\seq_1_device"

session = SessionManager(
    experiment_path,
    qop_ip="10.157.36.68",
    cluster_name="Cluster_2",
)

# Open QM connection and build element table (required before hardware ops)
session.open()

attr = session.attributes
cal = session.calibration

print(f"Resonator: {attr.ro_fq / 1e9:.4f} GHz")
print(f"Qubit:     {attr.qb_fq / 1e9:.4f} GHz")

## 0. OPX/Octave Mixer Calibration

Calibrate IQ mixer offsets (DC offset, gain imbalance, phase imbalance) for
each element. Run this before any measurements to ensure clean signal generation.

In [ ]:
hw = session.hw

# Gather elements and their frequencies
elements = [attr.ro_el, attr.qb_el]
el_los = hw.get_element_lo(elements)
el_ifs = [hw.calculate_el_if_fq(el, fq)
          for el, fq in zip(elements, [attr.ro_fq, attr.qb_fq])]

print("Mixer calibration targets:")
for el, lo, if_fq in zip(elements, el_los, el_ifs):
    print(f"  {el:20s}  LO={lo/1e9:.4f} GHz  IF={if_fq/1e6:.2f} MHz")

# Run Octave calibration for all elements
hw.calibrate_element(el=elements, target_LO=el_los, target_IF=el_ifs)
print("\nOctave mixer calibration complete.")

## 1. Full Readout Calibration Pipeline

Run the complete readout calibration: integration weights optimization,
GE discrimination, and butterfly QND measurement. Results are automatically
stored in `CalibrationStore`.

In [ ]:
cal_full = CalibrateReadoutFull(session)
result = cal_full.run(
    "readout",
    attr.qb_fq,
    r180="x180",
    n_samples_disc=50000,
    n_shots_butterfly=50000,
)

analysis = cal_full.analyze(result, update_calibration=True)
cal_full.plot(analysis)

print("\n--- Calibration Results ---")
print(f"Fidelity:  {analysis.metrics.get('ge_fidelity', 0):.2%}")
print(f"Angle:     {analysis.metrics.get('ge_angle', 0):.3f} rad")
print(f"Threshold: {analysis.metrics.get('ge_threshold', 0):.4f}")
print(f"F (QND):   {analysis.metrics.get('bfly_F', 0):.2%}")
print(f"Q:         {analysis.metrics.get('bfly_Q', 0):.2%}")
print(f"V:         {analysis.metrics.get('bfly_V', 0):.4f}")

In [ ]:
# Verify stored discrimination params
disc = cal.get_discrimination(attr.ro_el)
if disc:
    print(f"Stored angle:     {disc.angle:.4f}")
    print(f"Stored threshold: {disc.threshold:.4f}")
    print(f"Stored fidelity:  {disc.fidelity:.2%}")

## 2. Pulse-Train Tomography & Correction

Repeated gate applications reveal systematic amplitude and phase errors.
Use `fit_pulse_train()` to extract error parameters, then
`compute_corrected_knobs()` to derive corrected pulse parameters.

### 2.1 Run Pulse Train

In [ ]:
ptrain = QubitPulseTrain(session)
result = ptrain.run(
    N_values=list(range(1, 51)),
    rotation_pulse="x180",
    n_avg=5000,
)

analysis = ptrain.analyze(result)
ptrain.plot(analysis)
print(analysis.metrics)

### 2.2 Fit Pulse Train Errors

In [ ]:
# Extract raw data from result
S = result.output.extract("S")
N_values = result.output.extract("N_values")

if S is not None and N_values is not None:
    I_data = np.real(S)
    Q_data = np.imag(S)

    # Fit pulse-train model to extract errors
    pt_result = fit_pulse_train(
        N_values, I_data, Q_data,
        rotation_pulse="x180",
    )
    print(f"Amplitude error: {pt_result.amp_err:.4f}")
    print(f"Phase error:     {pt_result.phase_err:.4f} rad")
    print(f"Delta:           {pt_result.delta:.4f}")
    print(f"Zeta:            {pt_result.zeta:.4f}")

    # Store in calibration
    cal.set_pulse_train_result(attr.qb_el, pt_result)
    print("\nPulse train result saved to CalibrationStore.")

### 2.3 Compute Corrected Pulse Parameters

In [ ]:
# Retrieve the stored pulse-train result
stored_pt = cal.get_pulse_train_result(attr.qb_el)
if stored_pt:
    corrections = compute_corrected_knobs(stored_pt)
    print("Suggested corrections:")
    for key, val in corrections.items():
        print(f"  {key}: {val}")

    # Apply corrections (uncomment when ready):
    # pom = session.pulse_mgr
    # current_amp = pom.get_pulse_amplitude(attr.qb_el, "x180")
    # pom.set_pulse_amplitude(attr.qb_el, "x180", current_amp * corrections["amp_scale"])
    # session.burn_pulses()

### 2.4 Verify with AllXY

In [ ]:
allxy = AllXY(session)
result = allxy.run(n_avg=5000)

analysis = allxy.analyze(result)
allxy.plot(analysis)
print(f"Gate error = {analysis.metrics.get('gate_error', 'N/A'):.4f}")

## 3. AllXY Gate Error Diagnostics

The AllXY sequence is a sensitive diagnostic for pulse calibration errors.
Run it repeatedly to track gate error over time or after adjustments.

In [ ]:
import matplotlib.pyplot as plt

# Run AllXY multiple times to track stability
n_iterations = 5
gate_errors = []

allxy = AllXY(session)

for i in range(n_iterations):
    result = allxy.run(n_avg=5000)
    analysis = allxy.analyze(result)
    err = analysis.metrics.get("gate_error", np.nan)
    gate_errors.append(err)
    print(f"Iteration {i+1}: gate_error = {err:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_iterations + 1), gate_errors, "o-", ms=6)
ax.set_xlabel("Iteration")
ax.set_ylabel("Gate Error")
ax.set_title("AllXY Gate Error Stability")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Fock-Resolved SQR Calibration

Calibrate selective qubit rotation (SQR) gates for individual Fock manifolds.
Three approaches available:
- **Iterative** — gradient-free line search
- **SPSA** — simultaneous perturbation stochastic approximation

### 4.1 Fock-Resolved Power Rabi (Initial Calibration)

In [ ]:
# Determine Fock-dependent frequencies from number splitting
# fock_fqs = [...]  # From NumSplittingSpectroscopy results

fock_rabi = FockResolvedPowerRabi(session)
# result = fock_rabi.run(
#     fock_fqs=fock_fqs,
#     gains=np.linspace(0, 1.5, 50),
#     sel_qb_pulse="sel_x180",
#     disp_n_list=["disp_n0", "disp_n1", "disp_n2"],
#     n_avg=2000,
# )
#
# analysis = fock_rabi.analyze(result)
# fock_rabi.plot(analysis)
#
# # Store initial SQR calibration
# for key, val in analysis.metrics.items():
#     if key.startswith("g_pi_fock_"):
#         n = int(key.split("_")[-1])
#         sqr_cal = FockSQRCalibration(
#             fock_number=n,
#             model_type="power_rabi",
#             params={"g_pi": val},
#         )
#         print(f"Fock |{n}>: g_pi = {val:.4f}")

### 4.2 Iterative SQR Optimization

In [ ]:
# Example: optimize SQR gate for Fock |1>
#
# def measure_fidelity(params):
#     """Objective function: run Fock-resolved experiment and return fidelity."""
#     # Apply params to pulse manager
#     # Run Fock-resolved experiment
#     # Return scalar fidelity metric
#     pass
#
# best_params = optimize_fock_sqr_iterative(
#     measure_fidelity,
#     initial_params={"amplitude": 0.5, "frequency": fock_fqs[1]},
#     max_iters=20,
#     step_sizes={"amplitude": 0.01, "frequency": 100e3},
# )
# print(f"Optimized params: {best_params}")

### 4.3 SPSA SQR Optimization

In [ ]:
# SPSA is more efficient for noisy objectives with multiple parameters.
#
# best_params = optimize_fock_sqr_spsa(
#     measure_fidelity,
#     initial_params={"amplitude": 0.5, "frequency": fock_fqs[1], "phase": 0.0},
#     max_iters=50,
#     a=0.1,
#     c=0.05,
# )
# print(f"SPSA optimized params: {best_params}")

### 4.4 Store Final Calibration

In [ ]:
# After optimization, store the calibrated SQR parameters:
#
# from qubox_v2.calibration import FockSQRCalibration
#
# cals = [
#     FockSQRCalibration(
#         fock_number=0,
#         model_type="sqr",
#         params={"amplitude": best_n0_amp, "frequency": fock_fqs[0]},
#         fidelity=best_n0_fidelity,
#     ),
#     FockSQRCalibration(
#         fock_number=1,
#         model_type="sqr",
#         params={"amplitude": best_n1_amp, "frequency": fock_fqs[1]},
#         fidelity=best_n1_fidelity,
#     ),
# ]
# cal.set_fock_sqr_calibrations(attr.qb_el, cals)
# print("Fock SQR calibrations saved.")

---

## Summary

Calibration workflows use the same `run() -> analyze() -> plot()` protocol,
with `update_calibration=True` to persist results. The `CalibrationStore`
provides typed access to stored parameters:

```python
# Readout discrimination
disc = cal.get_discrimination(element)

# Coherence times
coh = cal.get_coherence(element)

# Pulse-train errors
pt = cal.get_pulse_train_result(element)

# Fock SQR calibrations
sqr = cal.get_fock_sqr_calibrations(element)
```

See `qubox_v2/calibration/algorithms.py` for all available calibration algorithms.